In [42]:
%pip install selenium webdriver-manager beautifulsoup4
%pip install undetected-chromedriver
%pip install pandas


from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup
from itertools import islice
import pandas as pd
import time
import requests
import os, csv, re

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Visits RT homepage to dismiss any potential overlay before navigating to reviews page, which can block access to reviews if not dismissed
def init_rt_session(driver):
    print("Initializing RT session to dismiss overlay...")
    driver.get("https://www.rottentomatoes.com")
    time.sleep(5)
    try:
        # Wait for overlay and click it
        close_btn = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-qa='close-overlay-btn']"))
        )
        driver.execute_script("arguments[0].click();", close_btn)
        time.sleep(2)
        print("Overlay dismissed.")
    except:
        print("No overlay found, continuing...")


def get_rt_urls(driver, movie_name, year=None):
    slug = movie_name.lower().replace(" ", "_")
    slug_no_article = re.sub(r'^(the|a|an)_', '', slug)

    for attempt_url in [
        f"https://www.rottentomatoes.com/m/{slug}_{year}",
        f"https://www.rottentomatoes.com/m/{slug}",
        f"https://www.rottentomatoes.com/m/{slug_no_article}_{year}",
        f"https://www.rottentomatoes.com/m/{slug_no_article}"
        
    ]:
        driver.get(f"{attempt_url}/reviews")
        time.sleep(3)

        try:
            close_btn = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "[data-qa='close-overlay-btn']"))
            )
            driver.execute_script("arguments[0].click();", close_btn)
            time.sleep(2)
        except:
            pass

        soup = BeautifulSoup(driver.page_source, "html.parser")
        has_reviews = len(soup.find_all("review-card")) > 0

        if has_reviews:
            critics_url = f"{attempt_url}/reviews"
            audience_url = f"{attempt_url}/reviews/all-audience"
            print(f"Found: {attempt_url}")
            return critics_url, audience_url
        else:
            print(f"No reviews at {attempt_url}, trying next...")

    raise ValueError(f"Could not find RT page for: {movie_name}")



# Scrapes top 50 critic reviews from RT
def scrape_rt_critic_reviews(driver, critics_url, movie_name):
    print("Scraping RT critic reviews...")
    driver.get(critics_url)
    time.sleep(5)

    try:
        close_btn = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-qa='close-overlay-btn']"))
        )
        driver.execute_script("arguments[0].click();", close_btn)
        time.sleep(3)
    except:
        pass

    driver.execute_script("window.scrollTo(0, 500);")
    time.sleep(3)

    reviews = []
    count = 1
    load_more_attempts = 0

    while len(reviews) < 50:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        review_cards = soup.find_all("review-card")

        for card in review_cards[len(reviews):]:  # Only process new cards
            if len(reviews) >= 50:
                break

            review = {}
            review["id"] = count
            count += 1

            score_tag = card.find("score-icon-critics")
            review["tomato_score"] = score_tag.get("sentiment") if score_tag else "N/A"

            # Review text
            text_tag = card.find("div", slot="review")
            if text_tag:
                # Remove the "Go to Full Review" link text
                full_review_link = text_tag.find("rt-link")
                if full_review_link:
                    full_review_link.decompose()
                review["review_text"] = text_tag.get_text(strip=True)
            else:
                review["review_text"] = "N/A"

            reviews.append(review)

        if load_more_attempts >= 5:  # Prevent infinite loop if "Load More" keeps appearing but no new reviews are loaded
            print("Reached maximum load more attempts. Stopping.")
            break


        if len(reviews) < 50:
            try:
                load_more = WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "[data-pagemediareviewsmanager='loadMoreBtn:click']"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", load_more)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", load_more)
                time.sleep(3)
                print("Loaded more critic reviews...")
                load_more_attempts += 1
            except:
                print("No more critic reviews to load.")
                break

        else:
            break

    reviews = reviews[:50]
    df = pd.DataFrame(reviews, columns=["id", "tomato_score", "review_text"])
    safe_name = re.sub(r'[^\w\s-]', '', movie_name).strip().replace(' ', '_')
    filepath = f"critic-reviews/{safe_name}_rt_audience_reviews.csv"
    df.to_csv(filepath, index=False)
    print(f"{len(df)} critic reviews") 


# Scrapes 50 most recent audience reviews from RT
def scrape_rt_audience_reviews(driver, audience_url, movie_name):
    # print("Scraping RT audience reviews...")
    driver.get(audience_url)
    time.sleep(5)

    try:
        close_btn = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-qa='close-overlay-btn']"))
        )
        driver.execute_script("arguments[0].click();", close_btn)
        time.sleep(3)
    except:
        pass

    driver.execute_script("window.scrollTo(0, 500);")
    time.sleep(3)

    reviews = []
    count = 0
    processed = 0  
    load_more_attempts = 0

    while len(reviews) < 50:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        review_cards = soup.find_all("review-card")

        # After finding all review_cards with BeautifulSoup, extract review texts via JS
        review_texts = driver.execute_script("""
            const cards = document.querySelectorAll('review-card');
            return Array.from(cards).map(card => {
                const drawerMore = card.querySelector('drawer-more');
                if (!drawerMore) return 'N/A';
                const content = drawerMore.querySelector('[slot="content"]');
                return content ? content.innerText.trim() : 'N/A';
            });
        """)

        for i, card in enumerate(review_cards):
            if len(reviews) >= 50:
                break

            if i < processed:  # Only process new cards
                continue

            review = {}
            count += 1
            review["id"] = count
            
            # name_tag = card.find("rt-link", slot="name")
            # review["reviewer_name"] = name_tag.get_text(strip=True) if name_tag else "N/A"

            # username_tag = card.find("span", slot="userhandle")
            # review["reviewer_username"] = username_tag.get_text(strip=True).lstrip("@") if username_tag else "N/A"

            # date_tag = card.find("span", slot="timestamp")
            # review["date"] = date_tag.get_text(strip=True) if date_tag else "N/A"

            rating_tag = card.find("rating-stars-group")
            review["rating"] = rating_tag.get("score") if rating_tag else "N/A"

            review["review_text"] = review_texts[i] if i < len(review_texts) else "N/A" 
            
            reviews.append(review)
            
        processed = len(reviews)  # Update processed count after processing current batch

        if load_more_attempts >= 5:  # Prevent infinite loop if "Load More" keeps appearing but no new reviews are loaded
            print("Reached maximum load more attempts. Stopping.")
            break

        if len(reviews) < 50:
            try:
                load_more = WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "[data-pagemediareviewsmanager='loadMoreBtn:click']"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", load_more)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", load_more)
                time.sleep(3)
                print("Loaded more audience reviews...")
                load_more_attempts += 1
            except:
                print("No more audience reviews to load.")
                break

    reviews = reviews[:50]
    df = pd.DataFrame(reviews, columns=["id", "rating", "review_text"])
    safe_name = re.sub(r'[^\w\s-]', '', movie_name).strip().replace(' ', '_')
    filepath = f"audience-reviews/{safe_name}_rt_audience_reviews.csv"
    df.to_csv(filepath, index=False)
    print(f"{len(df)} audience reviews")

In [45]:
movies = {
    'Shawshank Redemption': 1994,
    'Parasite': 2019,
    'star wars episode v the empire strikes back': 1980,
    'The Godfather': 1972,
    'Top Gun: Maverick': 2022,
    'The Godfather: Part II': 1974,
    'Spider-Man: Into the Spider-Verse': 2018,
    'Raiders of the Lost Ark': 1981,
    'The Dark Knight': 2008,
    'Mission: Impossible - Dead Reckoning Part One': 2023,
    '1000013 12 Angry Men': 1957,
    'Pulp Fiction': 1994,
    "Schindler's List": 1993,
    'Mad Max: Fury Road': 2015,
    'Harakiri': 1962,
    'The Lord of the Rings: The Return of the King': 2003,
    'Spirited Away': 2001,
    'Ikiru': 1952,
    'Inside Out': 2015,
    'The Godfather Part II': 1974,
    'The Lord of the Rings: The Fellowship of the Ring': 2001,
    'Coco': 2017,
    '1032176 Goodfellas': 1990,
    'The Good, the Bad and the Ugly': 1966,
    'Toy Story 3': 2010,
    'Rear Window': 1954,
    'Forrest Gump': 1994,
    'Toy Story 4': 2019,
    'City Lights': 1931,
    'Fight Club': 1999,
    'Finding Nemo': 2003,
    'Alien': 1979,
    'LOTR: The Two Towers': 2002,
    'How to Train Your Dragon': 2010,
    'The Silence of the Lambs': 1991,
    'Inception': 2010,
    'Up': 2009,
    'The Shining': 1980,
    'Star Wars: Empire Strikes Back': 1980,
    'Star Trek (2009)': 2009,
    'Se7en': 1995,
    'The Matrix': 1999,
    'The Social Network': 2010,
    'The Apartment': 1960,
    'Eighth Grade': 2018,
    'Casablanca': 1942,
    "One Flew Over the Cuckoo's Nest": 1975,
    'Argo': 2012,
    'Seven Samurai': 1954,
    'Portrait of a Lady on Fire': 2019,
    'Hell or High Water': 2016,
    "It's a Wonderful Life": 1946,
    'Memento': 2000,
    'Psycho': 1960,
    'Spider-Man: No Way Home': 2021,
    "Singin' in the Rain": 1952,
    'City of God': 2002,
    'The Farewell': 2019,
    'Apocalypse Now': 1979,
    'Saving Private Ryan': 1998,
    'Whiplash': 2014,
    'Vertigo': 1958,
    'Life Is Beautiful': 1997,
    'Your Name': 2016,
    'Citizen Kane': 1941,
    'The Green Mile': 1999,
    'The Pianist': 2002,
    'Taxi Driver': 1976,
    'Star Wars: A New Hope': 1977,
    'Harry Potter & Deathly Hallows Part 2': 2011,
    '2001: A Space Odyssey': 1968,
    'Interstellar': 2014,
    'Spider-Man: Across the Spider-Verse': 2023,
    'Dr. Strangelove': 1964,
    'Terminator 2: Judgment Day': 1991,
    'Get Out': 2017,
    'Double Indemnity': 1944,
    'Back to the Future': 1985,
    'Selma': 2014,
    'Raging Bull': 1980,
    'Moonlight': 2016,
    'Chinatown': 1974,
    'Lady Bird': 2017,
    'Lawrence of Arabia': 1962,
    'Dunkirk': 2017,
    'Sunset Boulevard': 1950,
    'Paddington 2': 2017,
    'Jaws': 1975,
    'Leon: The Professional': 1994,
    'Black Panther': 2018,
    'Some Like It Hot': 1959,
    'The Lion King': 1994,
    'In the Mood for Love': 2000,
    'Paths of Glory': 1957,
    'Gladiator': 2000,
    'Arrival': 2016,
    'American History X': 1998,
    'Inside Llewyn Davis': 2013,
    'Toy Story': 1995,
    'The Departed': 2006,
    'Marriage Story': 2019,
    'The Prestige': 2006,
    'Anatomy of a Fall': 2023,
    'The Third Man': 1949,
    'Boyhood': 2014,
    'The Wizard of Oz': 1939,
    'The Usual Suspects': 1995,
    'Gravity': 2013,
    'Roma': 2018,
    'Grave of the Fireflies': 1988,
    'The Holdovers': 2023,
    'Rashomon': 1950,
    'Everything Everywhere All at Once': 2022,
    'North by Northwest': 1959,
    'The Intouchables': 2011,
    'Minari': 2020,
    'Blade Runner': 1982,
    'Modern Times': 1936,
    'Past Lives': 2023,
    'Blue Velvet': 1986,
    'Once Upon a Time in the West': 1968,
    'Carol': 2015,
    '8-12': 1963,
    'If Beale Street Could Talk': 2018,
    'On the Waterfront': 1954,
    'Cinema Paradiso': 1988,
    'First Cow': 2019,
    'Amadeus': 1984,
    'Call Me by Your Name': 2017,
    'Do the Right Thing': 1989,
    'Widows': 2018,
    'Ran': 1985,
    '12 Years a Slave': 2013,
    'Metropolis': 1927,
    'The Wolf of Wall Street': 2013,
    'Unforgiven': 1992,
    'Shoplifters': 2018,
    'Django Unchained': 2012,
    'Brooklyn': 2015,
    'Touch of Evil': 1958,
    "Won't You Be My Neighbor?": 2018,
    'The Lives of Others': 2006,
    'The Favourite': 2018,
    'Stalker': 1979,
    'Sunset Blvd.': 1950,
    'A Separation': 2011,
    'Annie Hall': 1977,
    'Little Women (2019)': 2019,
    'Bridge on the River Kwai': 1957,
    'Phantom Thread': 2017,
    'Great Expectations': 1946,
    'The Great Dictator': 1940,
    'Tar': 2022,
    'The General': 1926,
    'Witness for the Prosecution': 1957,
    'Before Sunset': 2004,
    'Butch Cassidy & Sundance Kid': 1969,
    'Avengers: Infinity War': 2018,
    'The Shape of Water': 2017,
    'To Kill a Mockingbird': 1962,
    'Aliens': 1986,
    'The Handmaiden': 2016,
    'Stagecoach': 1939,
    'American Beauty': 1999,
    'Manchester by the Sea': 2016,
    'All About Eve': 1950,
    'Once Upon a Time in Hollywood': 2019,
    'General': 1926,
    'Burning': 2018,
    'Fargo': 1996,
    'Oldboy': 2003,
    'Under the Skin': 2013,
    'The 400 Blows': 1959,
    'Princess Mononoke': 1997,
    'Her': 2013,
    'Dark Knight Rises': 2012,
    'No Country for Old Men': 2007,
    'Ben-Hur': 1959,
    'The Grand Budapest Hotel': 2014,
    'Bonnie and Clyde': 1967,
    'La La Land': 2016,
    'Beauty and the Beast': 1946,
    'Braveheart': 1995,
    'Birdman': 2014,
    'Breathless': 1960,
    'Joker': 2019,
    'Spotlight': 2015,
    'Bicycle Thieves': 1948,
    'Ex Machina': 2014,
    'O Brother, Where Art Thou?': 2000,
    'Notorious': 1946,
    'Das Boot': 1981,
    'Mulholland Drive': 2001,
    'The Searchers': 1956,
    'Inglourious Basterds': 2009,
    "Pan's Labyrinth": 2006,
    'Hamilton': 2020,
    'Amélie': 2001,
    'Come and See': 1985,
    '3 Idiots': 2009,
    'Million Dollar Baby': 2004,
    'The Bridge on the River Kwai': 1957,
    'Brokeback Mountain': 2005,
    'A Clockwork Orange': 1971,
    'The Sting': 1973,
    'The Master': 2012,
    'Wild Strawberries': 1957,
    'Good Will Hunting': 1997,
    'Blade Runner 2049': 2017,
    'High Noon': 1952,
    'Requiem for a Dream': 2000,
    'The Deer Hunter': 1978,
    'Inside Out 2': 2024,
    'Eternal Sunshine of the Spotless Mind': 2004,
    'Avengers: Endgame': 2019,
    'Manhattan': 1979,
    'Wall-E': 2008,
    'The Great Escape': 1963,
    'John Wick': 2014,
    'LA Confidential': 1997,
    'Full Metal Jacket': 1987,
    'The Zone of Interest': 2023,
    'Aguirre, the Wrath of God': 1972,
    'Decision to Leave': 2022,
    'Strangers on a Train': 1951,
    'The Maltese Falcon': 1941,
    'Poor Things': 2023,
    'Heat': 1995,
    'Reservoir Dogs': 1992,
    'Kill Bill: Vol 1': 2003,
    'Barry Lyndon': 1975,
    'The Iron Claw': 2023,
    'Platoon': 1986,
    'Godzilla Minus One': 2023,
    'Seven': 1995,
    'Hunt for the Wilderpeople': 2016,
    'Bottoms': 2023,
    'The Wild Bunch': 1969,
    '1917': 2019,
    'Oppenheimer': 2023,
    'Rebecca': 1940,
    'Talk to Me': 2022,
    'The Treasure of the Sierra Madre': 1948,
    'The Wrestler': 2008,
    'Dog Day Afternoon': 1975,
}

def clean_movie_titles(movies):
    cleaned = {}
    for movie, year in movies.items():
        slug = movie.lower()
        slug = re.sub(r"[',:]", '', slug)        # remove apostrophes, commas, colons silently
        slug = re.sub(r'[^\w\s]', ' ', slug)  # replace all punctuation with space
        slug = re.sub(r'\s+', '_', slug.strip())  # replace all whitespace with single underscore
        cleaned[slug] = year
    return cleaned

cleaned_movies = clean_movie_titles(movies)
print(cleaned_movies)

{'shawshank_redemption': 1994, 'parasite': 2019, 'star_wars_episode_v_the_empire_strikes_back': 1980, 'the_godfather': 1972, 'top_gun_maverick': 2022, 'the_godfather_part_ii': 1974, 'spider_man_into_the_spider_verse': 2018, 'raiders_of_the_lost_ark': 1981, 'the_dark_knight': 2008, 'mission_impossible_dead_reckoning_part_one': 2023, '1000013_12_angry_men': 1957, 'pulp_fiction': 1994, 'schindlers_list': 1993, 'mad_max_fury_road': 2015, 'harakiri': 1962, 'the_lord_of_the_rings_the_return_of_the_king': 2003, 'spirited_away': 2001, 'ikiru': 1952, 'inside_out': 2015, 'the_lord_of_the_rings_the_fellowship_of_the_ring': 2001, 'coco': 2017, '1032176_goodfellas': 1990, 'the_good_the_bad_and_the_ugly': 1966, 'toy_story_3': 2010, 'rear_window': 1954, 'forrest_gump': 1994, 'toy_story_4': 2019, 'city_lights': 1931, 'fight_club': 1999, 'finding_nemo': 2003, 'alien': 1979, 'lotr_the_two_towers': 2002, 'how_to_train_your_dragon': 2010, 'the_silence_of_the_lambs': 1991, 'inception': 2010, 'up': 2009, 't

In [ ]:
def make_slug(title):
    replacements = {'é': 'e', 'è': 'e', 'ê': 'e', 'ë': 'e', 'à': 'a', 'â': 'a',
                    'ä': 'a', 'î': 'i', 'ï': 'i', 'ô': 'o', 'ö': 'o', 'ù': 'u',
                    'û': 'u', 'ü': 'u', 'ç': 'c', 'ñ': 'n', 'á': 'a', 'í': 'i',
                    'ó': 'o', 'ú': 'u', '½': '_1_2', '¼': '_1_4', '¾': '_3_4'}
    slug = title.lower()
    for char, rep in replacements.items():
        slug = slug.replace(char, rep)
    slug = re.sub(r"[',:]", '', slug)
    slug = re.sub(r'[^\w\s]', ' ', slug)
    slug = re.sub(r'\s+', '_', slug.strip())
    return slug

movies = {
    'Shawshank Redemption': 1994, 
    'Parasite': 2019,
    'star wars episode v the empire strikes back': 1980, 
    'The Godfather': 1972,
    'Top Gun: Maverick': 2022, 
    'The Godfather: Part II': 1974,
    'Spider-Man: Into the Spider-Verse': 2018, 
    'Raiders of the Lost Ark': 1981,
    'The Dark Knight': 2008, 
    'Mission: Impossible - Dead Reckoning Part One': 2023,
    '1000013 12 Angry Men': 1957, 
    'Pulp Fiction': 1994, 
    "Schindler's List": 1993,
    'Mad Max: Fury Road': 2015, 
    'Harakiri': 1962,
    'The Lord of the Rings: The Return of the King': 2003, 
    'Spirited Away': 2001,
    'Ikiru': 1952, 'Inside Out': 2015, 
    'The Godfather Part II': 1974,
    'The Lord of the Rings: The Fellowship of the Ring': 2001, 
    'Coco': 2017,
    'Goodfellas': 1990, 
    'The Good, the Bad and the Ugly': 1966, 
    'Toy Story 3': 2010,
    'Rear Window': 1954, 
    'Forrest Gump': 1994, 
    'Toy Story 4': 2019, 
    'City Lights': 1931,
    'Fight Club': 1999, 
    'Finding Nemo': 2003, 
    'Alien': 1979, 
    'LOTR: The Two Towers': 2002,
    'How to Train Your Dragon': 2010, 
    'The Silence of the Lambs': 1991, 
    'Inception': 2010,
    'Up': 2009, 'The Shining': 1980, 
    'Star Wars: Empire Strikes Back': 1980,
    'Star Trek (2009)': 2009, 
    'Se7en': 1995, 
    'The Matrix': 1999, 
    'The Social Network': 2010,
    'The Apartment': 1960, 
    'Eighth Grade': 2018, 
    'Casablanca': 1942,
    "One Flew Over the Cuckoo's Nest": 1975, 
    'Argo': 2012, 'Seven Samurai': 1954,
    'Portrait of a Lady on Fire': 2019, 
    'Hell or High Water': 2016,
    "It's a Wonderful Life": 1946, 
    'Memento': 2000, 
    'Psycho': 1960,
    'Spider-Man: No Way Home': 2021, 
    "Singin' in the Rain": 1952, 
    'City of God': 2002,
    'The Farewell': 2019, 
    'Apocalypse Now': 1979, 
    'Saving Private Ryan': 1998,
    'Whiplash': 2014, 'Vertigo': 1958, 'Life Is Beautiful': 1997, 
    'Your Name': 2016,
    'Citizen Kane': 1941, 
    'The Green Mile': 1999, 
    'The Pianist': 2002, 
    'Taxi Driver': 1976,
    'Star Wars: A New Hope': 1977, 
    'Harry Potter & Deathly Hallows Part 2': 2011,
    '2001: A Space Odyssey': 1968, 
    'Interstellar': 2014,
    'Spider-Man: Across the Spider-Verse': 2023, 
    'Dr. Strangelove': 1964,
    'Terminator 2: Judgment Day': 1991, 
    'Get Out': 2017, 
    'Double Indemnity': 1944,
    'Back to the Future': 1985, 
    'Selma': 2014, 
    'Raging Bull': 1980, 
    'Moonlight': 2016,
    'Chinatown': 1974, 
    'Lady Bird': 2017, 
    'Lawrence of Arabia': 1962, 
    'Dunkirk': 2017,
    'Sunset Boulevard': 1950, 
    'Paddington 2': 2017, 
    'Jaws': 1975,
    'Leon: The Professional': 1994, 
    'Black Panther': 2018, 
    'Some Like It Hot': 1959,
    'The Lion King': 1994, 'In the Mood for Love': 2000, 'Paths of Glory': 1957,
    'Gladiator': 2000, 'Arrival': 2016, 'American History X': 1998,
    'Inside Llewyn Davis': 2013, 'Toy Story': 1995, 'The Departed': 2006,
    'Marriage Story': 2019, 'The Prestige': 2006, 'Anatomy of a Fall': 2023,
    'The Third Man': 1949, 'Boyhood': 2014, 'The Wizard of Oz': 1939,
    'The Usual Suspects': 1995, 'Gravity': 2013, 'Roma': 2018,
    'Grave of the Fireflies': 1988, 'The Holdovers': 2023, 'Rashomon': 1950,
    'Everything Everywhere All at Once': 2022, 'North by Northwest': 1959,
    'The Intouchables': 2011, 'Minari': 2020, 'Blade Runner': 1982, 'Modern Times': 1936,
    'Past Lives': 2023, 'Blue Velvet': 1986, 'Once Upon a Time in the West': 1968,
    'Carol': 2015, '8-12': 1963, 'If Beale Street Could Talk': 2018,
    'On the Waterfront': 1954, 'Cinema Paradiso': 1988, 'First Cow': 2019,
    'Amadeus': 1984, 'Call Me by Your Name': 2017, 'Do the Right Thing': 1989,
    'Widows': 2018, 'Ran': 1985, '12 Years a Slave': 2013, 'Metropolis': 1927,
    'The Wolf of Wall Street': 2013, 'Unforgiven': 1992, 'Shoplifters': 2018,
    'Django Unchained': 2012, 'Brooklyn': 2015, 'Touch of Evil': 1958,
    "Won't You Be My Neighbor?": 2018, 'The Lives of Others': 2006, 'The Favourite': 2018,
    'Stalker': 1979, 'Sunset Blvd.': 1950, 'A Separation': 2011, 'Annie Hall': 1977,
    'Little Women (2019)': 2019, 'Bridge on the River Kwai': 1957, 'Phantom Thread': 2017,
    'Great Expectations': 1946, 'The Great Dictator': 1940, 'Tar': 2022,
    'The General': 1926, 'Witness for the Prosecution': 1957, 'Before Sunset': 2004,
    'Butch Cassidy & Sundance Kid': 1969, 'Avengers: Infinity War': 2018,
    'The Shape of Water': 2017, 'To Kill a Mockingbird': 1962, 'Aliens': 1986,
    'The Handmaiden': 2016, 
    'Stagecoach': 1939, 
    'American Beauty': 1999,
    'Manchester by the Sea': 2016, 
    'All About Eve': 1950,
    'Once Upon a Time in Hollywood': 2019, 
    'General': 1926, 
    'Burning': 2018,
    'Fargo': 1996, 'Oldboy': 2003, 'Under the Skin': 2013, 'The 400 Blows': 1959,
    'Princess Mononoke': 1997, 'Her': 2013, 'Dark Knight Rises': 2012,
    'No Country for Old Men': 2007, 'Ben-Hur': 1959, 'The Grand Budapest Hotel': 2014,
    'Bonnie and Clyde': 1967, 'La La Land': 2016, 'Beauty and the Beast': 1946,
    'Braveheart': 1995, 'Birdman': 2014, 'Breathless': 1960, 'Joker': 2019,
    'Spotlight': 2015, 'Bicycle Thieves': 1948, 'Ex Machina': 2014,
    'O Brother, Where Art Thou?': 2000, 'Notorious': 1946, 'Das Boot': 1981,
    'Mulholland Drive': 2001, 'The Searchers': 1956, 'Inglourious Basterds': 2009,
    "Pan's Labyrinth": 2006, 'Hamilton': 2020, 'Amélie': 2001, 'Come and See': 1985,
    '3 Idiots': 2009, 'Million Dollar Baby': 2004, 'The Bridge on the River Kwai': 1957,
    'Brokeback Mountain': 2005, 'A Clockwork Orange': 1971, 'The Sting': 1973,
    'The Master': 2012, 'Wild Strawberries': 1957, 'Good Will Hunting': 1997,
    'Blade Runner 2049': 2017, 
    'High Noon': 1952, 
    'Requiem for a Dream': 2000,
    'The Deer Hunter': 1978, 
    'Inside Out 2': 2024,
    'Eternal Sunshine of the Spotless Mind': 2004, 
    'Avengers: Endgame': 2019,
    'Manhattan': 1979, 
    'Wall-E': 2008, 
    'The Great Escape': 1963, 
    'John Wick': 2014,
    'LA Confidential': 1997, 
    'Full Metal Jacket': 1987, 
    'The Zone of Interest': 2023,
    'Aguirre, the Wrath of God': 1972, 
    'Decision to Leave': 2022,
    'Strangers on a Train': 1951, 
    'The Maltese Falcon': 1941, 
    'Poor Things': 2023,
    'Heat': 1995, 
    'Reservoir Dogs': 1992, 
    'Kill Bill: Vol 1': 2003, 
    'Barry Lyndon': 1975,
    'The Iron Claw': 2023, 
    'Platoon': 1986, 
    'Godzilla Minus One': 2023, 
    'Seven': 1995,
    'Hunt for the Wilderpeople': 2016, 
    'Bottoms': 2023, 
    'The Wild Bunch': 1969,
    '1917': 2019, 
    'Oppenheimer': 2023, 
    'Rebecca': 1940, 
    'Talk to Me': 2022,
    'The Treasure of the Sierra Madre': 1948, 
    'The Wrestler': 2008, 
    'Dog Day Afternoon': 1975,
}

rt_urls = {movie: f"https://www.rottentomatoes.com/m/{make_slug(movie)}" 
           for movie in movies}

In [32]:
try:
    driver.quit()
except:
    pass

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

init_rt_session(driver)  # Call once to set the cookie

count = 0

for movie, year in islice(cleaned_movies.items(), 21, None):
    
    critics_url, audience_url = get_rt_urls(driver, movie, year)
    scrape_rt_critic_reviews(driver, critics_url, movie)
    scrape_rt_audience_reviews(driver, audience_url, movie)

    count += 1
    print(f"Processed movie: {movie} (Count: {count})")
    


driver.quit()

Initializing RT session to dismiss overlay...
Overlay dismissed.
No reviews at https://www.rottentomatoes.com/m/1032176_goodfellas_1990, trying next...
No reviews at https://www.rottentomatoes.com/m/1032176_goodfellas, trying next...
No reviews at https://www.rottentomatoes.com/m/1032176_goodfellas_1990, trying next...
No reviews at https://www.rottentomatoes.com/m/1032176_goodfellas, trying next...


ValueError: Could not find RT page for: 1032176_goodfellas

In [59]:
try:
    driver.quit()
except:
    pass

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

init_rt_session(driver)

count = 0
problematic_links = []

for movie, rt_base_url in islice(rt_urls.items(), 235, None):
    try:
        # Derive critics and audience URLs from the base RT url
        critics_url = rt_base_url + "/reviews"
        audience_url = rt_base_url + "/reviews/all-audience"

        # Check if the page exists
        driver.get(critics_url)
        time.sleep(3)

        try:
            close_btn = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "[data-qa='close-overlay-btn']"))
            )
            driver.execute_script("arguments[0].click();", close_btn)
            time.sleep(2)
        except:
            pass

        count += 1

        print(f"Checking for reviews for {movie}... (Count: {count})")

        soup = BeautifulSoup(driver.page_source, "html.parser")
        has_reviews = len(soup.find_all("review-card")) > 0
        if has_reviews:
            critic_reviews = scrape_rt_critic_reviews(driver, critics_url, movie)
            audience_reviews = scrape_rt_audience_reviews(driver, audience_url, movie)
            
            
        else:
            print(f"No reviews found for {movie}: {rt_base_url}")
            problematic_links.append({"movie": movie, "url": rt_base_url, "reason": "No reviews found"})
            
    except Exception as e:
        print(f"Error processing {movie} ({rt_base_url}): {e}")
        problematic_links.append({"movie": movie, "url": rt_base_url, "reason": str(e)})
        continue

print(f"\nDone. Successfully processed: {count} movies")
print(f"Problematic links ({len(problematic_links)}):")
for item in problematic_links:
    print(f"  - {item['movie']}: {item['url']} | Reason: {item['reason']}")

Initializing RT session to dismiss overlay...
Overlay dismissed.
Checking for reviews for Hunt for the Wilderpeople... (Count: 1)
Scraping RT critic reviews...
Loaded more critic reviews...
Loaded more critic reviews...
Saved 50 critic reviews for Hunt for the Wilderpeople
Loaded more audience reviews...
Loaded more audience reviews...
Saved 50 audience reviews for Hunt for the Wilderpeople
Checking for reviews for Bottoms... (Count: 2)
Scraping RT critic reviews...
Loaded more critic reviews...
Loaded more critic reviews...
Saved 50 critic reviews for Bottoms
Loaded more audience reviews...
Loaded more audience reviews...
Saved 50 audience reviews for Bottoms
Checking for reviews for The Wild Bunch... (Count: 3)
No reviews found for The Wild Bunch: https://www.rottentomatoes.com/m/the_wild_bunch
Checking for reviews for 1917... (Count: 4)
No reviews found for 1917: https://www.rottentomatoes.com/m/1917
Checking for reviews for Oppenheimer... (Count: 5)
No reviews found for Oppenheimer